# 02 - Featurisation sanity checks

Quick visual checks that the Phase 2 featurisers produce sensible outputs before we wire them into a baseline or a GNN.

Three sections:

1. **Ligand featurisation** — works with no real data (SMILES + RDKit only)
2. **Pocket extraction** — on a few real PDBbind complexes if they're downloaded; otherwise on a tiny synthetic structure
3. **ESM-2 embedding** — runs the 35M model on a single short sequence (downloads ~150 MB the first time)

Sections that need PDBbind data are clearly flagged and skip cleanly if `data/raw/` is empty.

In [ ]:
from pathlib import Path

import numpy as np

from plb.data import (
    ecfp_fingerprint,
    feature_dims,
    find_default_paths,
    ligand_graph_from_sdf,
    ligand_graph_from_smiles,
    pocket_from_pdb_files,
)
from plb.data.protein import ESMEmbedder, pool_pocket_embedding

DATA_ROOT = Path("../data")
paths = find_default_paths(DATA_ROOT)
have_pdbbind = paths["refined_root"].is_dir()
print(f"PDBbind refined-set found: {have_pdbbind}")
print(f"feature_dims = {feature_dims()}")

## 1. Ligand featurisation

Three reference molecules. We confirm atom counts, edge symmetry, and that the ECFP4 fingerprint produces clearly distinct bit patterns.

In [ ]:
examples = {
    "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "caffeine": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "benzene": "c1ccccc1",
    "methane": "C",
}

for name, smiles in examples.items():
    g = ligand_graph_from_smiles(smiles)
    fp = ecfp_fingerprint(smiles)
    print(
        f"{name:>10} | atoms={g.num_atoms:>3} | edges={g.num_edges:>3} | "
        f"ECFP bits set={int(fp.sum()):>3} | smiles={g.smiles}"
    )

In [ ]:
fp_aspirin = ecfp_fingerprint(examples["aspirin"])
fp_benzene = ecfp_fingerprint(examples["benzene"])
fp_caffeine = ecfp_fingerprint(examples["caffeine"])

def tanimoto(a, b):
    intersect = int(np.bitwise_and(a, b).sum())
    union = int(np.bitwise_or(a, b).sum())
    return intersect / max(1, union)

print(f"Tanimoto(aspirin, benzene)  = {tanimoto(fp_aspirin, fp_benzene):.3f}")
print(f"Tanimoto(aspirin, caffeine) = {tanimoto(fp_aspirin, fp_caffeine):.3f}")
print(f"Tanimoto(aspirin, aspirin)  = {tanimoto(fp_aspirin, fp_aspirin):.3f}")

## 2. Pocket extraction

If PDBbind is downloaded, we run on three well-known complexes (a kinase, a serine protease, an HIV protease). Otherwise we fall back to the tiny synthetic structure used in the unit tests.

In [ ]:
EXAMPLE_PDB_IDS = ["1a30", "2qbr", "3ptb"]  # adjust to whatever's downloaded

if have_pdbbind:
    refined_root = paths["refined_root"]
    for pdb_id in EXAMPLE_PDB_IDS:
        folder = refined_root / pdb_id
        protein = folder / f"{pdb_id}_protein.pdb"
        ligand = folder / f"{pdb_id}_ligand.sdf"
        if not protein.is_file() or not ligand.is_file():
            print(f"{pdb_id}: not in your refined set, skipping")
            continue
        pocket = pocket_from_pdb_files(protein, ligand, cutoff_angstrom=6.0)
        print(
            f"{pdb_id} | chains={pocket.chain_ids} | total_residues={pocket.n_total_residues} | "
            f"pocket_residues={pocket.n_pocket_residues}"
        )
else:
    print("PDBbind data not found - skipping. Run scripts/download_pdbbind.py first.")

In [ ]:
# Synthetic fallback - works with no real data, mirrors the unit test
import tempfile

SYNTHETIC_PDB = """\
ATOM      1  N   ALA A   1       0.000   0.000   0.000  1.00 20.00           N
ATOM      2  CA  ALA A   1       1.500   0.000   0.000  1.00 20.00           C
ATOM      3  C   ALA A   1       2.000   1.500   0.000  1.00 20.00           C
ATOM      4  O   ALA A   1       3.000   2.000   0.000  1.00 20.00           O
ATOM      5  N   GLY A   2       1.500   2.500   0.000  1.00 20.00           N
ATOM      6  CA  GLY A   2       2.000   4.000   0.000  1.00 20.00           C
ATOM      7  C   GLY A   2       3.500   4.500   0.000  1.00 20.00           C
ATOM      8  O   GLY A   2       4.000   5.500   0.000  1.00 20.00           O
ATOM      9  N   VAL A   3      20.000  20.000  20.000  1.00 20.00           N
ATOM     10  CA  VAL A   3      21.500  20.000  20.000  1.00 20.00           C
ATOM     11  C   VAL A   3      22.000  21.500  20.000  1.00 20.00           C
ATOM     12  O   VAL A   3      23.000  22.000  20.000  1.00 20.00           O
END
"""

SYNTHETIC_SDF = """\
ligand
     RDKit          3D

  1  0  0  0  0  0  0  0  0  0999 V2000
    1.0000    1.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
M  END
$$$$
"""

with tempfile.TemporaryDirectory() as td:
    pdb = Path(td) / "tiny.pdb"
    sdf = Path(td) / "tiny.sdf"
    pdb.write_text(SYNTHETIC_PDB)
    sdf.write_text(SYNTHETIC_SDF)
    pocket = pocket_from_pdb_files(pdb, sdf, cutoff_angstrom=6.0)
    print(f"synthetic | chain={pocket.chain_ids} | seq={pocket.chains}")
    print(f"          | pocket_positions={pocket.pocket_residues}")
    print(f"          | -> {[pocket.chains[c][i] for c in pocket.chain_ids for i in pocket.pocket_residues[c]]}")

## 3. ESM-2 35M embedding

Run the 35M model on one short sequence end-to-end. The first execution will download the weights (~150 MB) and cache them under `~/.cache/torch/hub/`. Subsequent runs are instant.

Set `RUN_ESM = False` if you want to skip the download (e.g. on a slow connection).

In [ ]:
RUN_ESM = True

if RUN_ESM:
    embedder = ESMEmbedder(model_name="esm2_t12_35M_UR50D", device="cpu")
    sequence = "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEK"
    emb = embedder.embed_chain(sequence)
    print(f"sequence length: {len(sequence)}")
    print(f"embedding shape: {emb.shape} (expected ({len(sequence)}, {embedder.embed_dim}))")
    print(f"embedding dtype: {emb.dtype}")
    print(f"sample residue 0 (first 8 dims): {emb[0, :8]}")
else:
    print("RUN_ESM=False - skipping ESM model download.")

### Pocket pooling end-to-end

Combine sections 2 and 3: real PDBbind complex (or synthetic fallback) -> pocket residues -> ESM embedding -> pocket-pooled vector.

In [ ]:
if RUN_ESM:
    if have_pdbbind:
        pdb_id = EXAMPLE_PDB_IDS[0]
        folder = paths["refined_root"] / pdb_id
        protein = folder / f"{pdb_id}_protein.pdb"
        ligand = folder / f"{pdb_id}_ligand.sdf"
        if protein.is_file() and ligand.is_file():
            pocket = pocket_from_pdb_files(protein, ligand)
            print(f"Using real complex {pdb_id}")
        else:
            pocket = None
    else:
        pocket = None

    if pocket is None:
        # Use the synthetic mini-protein from above
        with tempfile.TemporaryDirectory() as td:
            pdb = Path(td) / "tiny.pdb"
            sdf = Path(td) / "tiny.sdf"
            pdb.write_text(SYNTHETIC_PDB)
            sdf.write_text(SYNTHETIC_SDF)
            pocket = pocket_from_pdb_files(pdb, sdf, cutoff_angstrom=6.0)
        print("Using synthetic 3-residue protein")

    chain_embeddings = {cid: embedder.embed_chain(seq) for cid, seq in pocket.chains.items()}
    pocket_vec, whole_vec = pool_pocket_embedding(chain_embeddings, pocket.pocket_residues, embedder.embed_dim)

    print(f"chains: {pocket.chain_ids}")
    print(f"pocket_residues: {pocket.n_pocket_residues} of {pocket.n_total_residues}")
    print(f"pocket_pool: {pocket_vec.shape} {pocket_vec.dtype}")
    print(f"whole_pool : {whole_vec.shape} {whole_vec.dtype}")
    print(f"L2(pocket - whole) = {np.linalg.norm(pocket_vec - whole_vec):.3f}")

## Next

Once `scripts/precompute_esm.py` has run over the whole refined set (~5316 complexes, a few hours on CPU or ~10 minutes on a 6 GB GPU), Phase 3 wires up the XGBoost baseline notebook on top of the cached embeddings + ECFP fingerprints.